# B07 · Streamlit 交互应用

> 阶段〇第 7 周。前六周的结果都躺在 notebook 里；本周用 Streamlit 把仿真包成
> **浏览器里可调参的 Web App**——不需要任何前端知识，纯 Python。
> 这就是以后给导师/同事演示 RL 策略效果、给 Isaac 仿真做调参面板的技能原型。

## 学习目标

1. 理解 Streamlit 心智模型：**脚本自上而下重跑**，widget 的值就是 Python 变量；
2. 使用 `st.slider / st.selectbox / st.line_chart / st.metric / st.columns` 等核心组件；
3. 理解 `st.session_state`：重跑之间需要保留的状态放这里；
4. 从 notebook 生成一个完整可运行的 `app.py`（一阶环节 + PID 调参面板）；
5. 掌握「语法检查 + headless 冒烟测试」的自动化验证套路。

> 说明：Streamlit App 是**独立的 Python 脚本**，在终端用 `streamlit run` 启动，
> 不能在 notebook 内部运行（notebook 的 kernel 不是网页服务器）。
> 本 notebook 负责：生成 app.py → 验证语法 → 冒烟测试证明它能正常启动。

## 1. Streamlit 心智模型（和 notebook 完全不同）

| | Jupyter notebook | Streamlit |
|---|---|---|
| 执行方式 | 逐 cell 手动执行，可乱序 | **每次交互，整个脚本自上而下重跑** |
| 状态保存 | 变量留在 kernel 里 | 重跑后普通变量全部重来；要保留用 `st.session_state` |
| 输出 | cell 下方 | `st.write / st.line_chart / st.pyplot` 渲染到网页 |
| 输入 | 改代码 | widget（滑块、下拉框…），**widget 调用即返回值** |

最小例子（写在 app.py 里，不要在 notebook 里 import streamlit 跑）：

```python
import streamlit as st
Kp = st.slider("Kp", 0.0, 10.0, 2.0)   # 拖动滑块 → 整个脚本重跑 → Kp 变成新值
st.write("当前 Kp =", Kp)
```

**关键理解**：没有回调、没有前端代码。你写的是一个「每次重跑都根据 widget 当前值重新计算并渲染」的纯 Python 脚本。
`st.session_state` 是一个跨重跑保留的字典式对象，存「计数器、历史结果」这类需要记忆的东西。

## 2. 生成完整 app.py：一阶惯性环节 + PID 调参面板

功能设计：

- 侧栏滑块：被控对象参数（增益 $K$、时间常数 $\tau$）、PID 三参数、设定值 $r$；
- 主区两列：阶跃响应曲线 + 控制量曲线（`st.line_chart`）；
- 三个指标卡：超调量、调节时间、稳态误差（`st.metric`）；
- 一个 `st.session_state` 演示按钮。

仿真用固定步长欧拉法（B01 的手艺），闭环结构：
$e = r - y$，$u = K_p e + K_i \int e\,dt + K_d \dot{e}$，对象 $\tau\dot{y} + y = Ku$。

下面这个 cell 把 app 写入 `runs/00_basic/b07/app.py`：

In [1]:
from pathlib import Path

PROJECT_ROOT = next(p for p in Path.cwd().parents if (p / "pyproject.toml").exists())
APP_DIR = PROJECT_ROOT / "runs" / "00_basic" / "b07"
APP_DIR.mkdir(parents=True, exist_ok=True)
APP = APP_DIR / "app.py"

APP_SOURCE = '''
# app.py -- 一阶惯性环节 + PID 调参面板
# 启动方式（终端）: uv run streamlit run runs/00_basic/b07/app.py
import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="PID Tuning Panel", layout="wide")
st.title("First-order plant + PID tuning panel")
st.caption("Plant: tau*dy/dt + y = K*u;  Controller: u = Kp*e + Ki*integral(e) + Kd*de/dt")

# ---- 侧栏：参数区 ----
st.sidebar.header("Plant parameters")
K = st.sidebar.slider("Plant gain K", 0.5, 5.0, 2.0, 0.1)
tau = st.sidebar.slider("Time constant tau (s)", 0.1, 3.0, 0.5, 0.1)
r = st.sidebar.number_input("Setpoint r", value=1.0, step=0.1)

st.sidebar.header("PID gains")
Kp = st.sidebar.slider("Kp", 0.0, 10.0, 2.0, 0.1)
Ki = st.sidebar.slider("Ki", 0.0, 10.0, 2.0, 0.1)
Kd = st.sidebar.slider("Kd", 0.0, 2.0, 0.05, 0.05)

# ---- 仿真：固定步长欧拉法 ----
def simulate(K, tau, Kp, Ki, Kd, r, t_end=10.0, dt=0.005):
    n = int(t_end / dt) + 1
    t = np.linspace(0, t_end, n)
    y = np.zeros(n)
    u = np.zeros(n)
    int_e = 0.0
    e_prev = r
    for k in range(1, n):
        e = r - y[k - 1]
        int_e += e * dt
        d_e = (e - e_prev) / dt
        e_prev = e
        u[k] = Kp * e + Ki * int_e + Kd * d_e
        y[k] = y[k - 1] + dt * (K * u[k] - y[k - 1]) / tau
    return t, y, u

t, y, u = simulate(K, tau, Kp, Ki, Kd, r)

# ---- 指标计算 ----
y_ss = y[-1]
overshoot = max(0.0, (y.max() - y_ss) / max(abs(y_ss), 1e-9) * 100)
band = 0.02 * max(abs(y_ss), 1e-9)
outside = np.where(np.abs(y - y_ss) > band)[0]
t_settle = float(t[outside[-1]]) if len(outside) else 0.0
ess = r - y_ss

m1, m2, m3 = st.columns(3)
m1.metric("Overshoot (%)", f"{overshoot:.1f}")
m2.metric("Settling time (s)", f"{t_settle:.2f}")
m3.metric("Steady-state error", f"{ess:.4f}")

# ---- 图表区 ----
c1, c2 = st.columns(2)
with c1:
    st.subheader("Step response")
    st.line_chart(pd.DataFrame({"y": y, "setpoint": r}, index=t))
with c2:
    st.subheader("Control effort u")
    st.line_chart(pd.DataFrame({"u": u}, index=t))

# ---- session_state 演示：跨重跑保留的计数器 ----
if "tune_count" not in st.session_state:
    st.session_state.tune_count = 0
if st.button("Mark this tuning round"):
    st.session_state.tune_count += 1
st.caption(f"session_state demo: you have marked {st.session_state.tune_count} tuning round(s). "
           "This number survives reruns; plain variables do not.")
'''

APP.write_text(APP_SOURCE, encoding="utf-8")
print("已生成:", APP)
print("行数:", len(APP_SOURCE.splitlines()))

已生成: /data/wangf/robot_rl_learn/runs/00_basic/b07/app.py
行数: 70


## 3. 启动命令（请在终端运行，不在 notebook 内执行）

> **运行前提**：项目 `.venv` 已 `uv sync`；上一步 cell 已生成 `app.py`。
> 在本机终端执行（注意：不要在本 notebook 里运行——它会阻塞 kernel）：

```bash
cd /data/wangf/robot_rl_learn
uv run streamlit run runs/00_basic/b07/app.py
```

> **预期输出**：终端打印 `You can now view your Streamlit app in your browser.` 和
> `Local URL: http://localhost:8501`；浏览器打开后左侧是参数滑块，
> 拖动 **Kp / Ki / Kd** 滑块，右侧阶跃响应曲线与三个指标卡实时更新。
> 试试：Ki=0 时有稳态误差（P/PD 控制的静差）；Ki 增大后静差消除但超调变大。
> 按 `Ctrl+C` 停止服务。

## 4. 自动化验证 I：语法检查

不启动服务，先用 `py_compile` 证明 app.py 语法正确——CI 里检查脚本的第一步。

In [2]:
import py_compile

py_compile.compile(str(APP), doraise=True)   # 语法错误会抛异常
print("语法检查通过:", APP.name)

语法检查通过: app.py


## 5. 自动化验证 II：headless 冒烟测试

以无头模式（`--server.headless true`，不自动开浏览器）启动 streamlit，
轮询其 `/healthz` 健康端点，确认服务正常拉起后再终止——
这就是「启动几秒后 terminate 并检查无报错」的标准冒烟测试。
若当前环境限制端口/子进程，则退化为「语法检查 + 启动输出检查」。

In [3]:
import subprocess
import sys
import time
import urllib.request

PORT = 8765
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", str(APP),
     "--server.headless", "true",
     "--server.port", str(PORT),
     "--server.address", "127.0.0.1"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

healthy = False
try:
    for _ in range(40):                      # 最多等 40 s
        time.sleep(1)
        if proc.poll() is not None:          # 进程提前退出 = 启动失败
            break
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/healthz", timeout=1) as resp:
                healthy = resp.status == 200
                if healthy:
                    break
        except Exception:
            continue                        # 服务还没起来，继续等
finally:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()

if healthy:
    print(f"冒烟测试通过：streamlit 正常启动，/healthz 返回 200，进程已干净退出（返回码 {proc.returncode}）")
else:
    output = proc.stdout.read() if proc.stdout else ""
    print("冒烟测试未通过或环境受限。返回码:", proc.returncode)
    print("启动输出（前 1500 字符）:\n", output[:1500])

冒烟测试通过：streamlit 正常启动，/healthz 返回 200，进程已干净退出（返回码 0）


## 小结与衔接

- Streamlit = 「交互一次、全脚本重跑」+ widget 即变量 + `session_state` 管记忆；
- 开发套路：notebook 里调通算法 → 拷进 app.py → `py_compile` 语法检查 → headless 冒烟测试；
- 调参面板的价值：**把「改参数-重跑-看图」的循环从分钟级压到秒级**。

**下周 B08（阶段项目）**：把 B01–B07 全部串起来——二阶系统上 P/PI/PID 对比仿真，
指标计算、报告图、交互图、代码结构、部署指引，一次走完。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。

**练习 1（★，15 分钟，10 分）——加对象类型选择**
给 app.py 加一个 `st.selectbox`：被控对象可选「一阶惯性环节」或「积分环节」（$\dot{y} = Ku$）。
积分环节无稳态（输出持续爬升），P 控制下观察现象。
**交付物**：修改后的代码片段（markdown 代码块即可）+ 现象描述一句话。

**练习 2（★，10 分钟，10 分）——限幅（饱和）**
真实执行器有输出上限。在 `simulate` 里给 `u` 加限幅 `u[k] = np.clip(uk, -10, 10)`，
再用滑块加一个大 Ki（如 8）观察：响应出现什么异常？（这是 **积分饱和（windup）** 的前奏。）
**交付物**：修改片段 + 观察描述。

**练习 3（★★，20 分钟，15 分）——下载按钮**
加一个 `st.download_button`：把当前仿真结果 `(t, y, u)` 导出为 CSV 供下载
（提示：`pd.DataFrame(...).to_csv().encode("utf-8")` 作为 data 参数）。
**交付物**：代码片段。

**练习 4（★，5 分钟，10 分）——概念问答**
为什么「上次仿真结果」这种变量必须放 `st.session_state`，普通全局变量不行？
**交付物**：markdown 两句话。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：

```python
plant_type = st.sidebar.selectbox("Plant type", ["first-order", "integrator"])
# simulate 中：
if plant_type == "integrator":
    y[k] = y[k - 1] + dt * K * u[k]
else:
    y[k] = y[k - 1] + dt * (K * u[k] - y[k - 1]) / tau
```

现象：积分环节在 P 控制下也能无静差（对象本身含一个积分环节，系统型别为 1 型）。

**练习 2**：`u[k] = float(np.clip(uk, -10.0, 10.0))`。
大 Ki + 限幅时：误差长期不消，积分项持续累积，退出饱和后输出猛烈过冲、大幅振荡——积分饱和。

**练习 3**：

```python
csv_bytes = pd.DataFrame({"t": t, "y": y, "u": u}).to_csv(index=False).encode("utf-8")
st.download_button("Download result CSV", data=csv_bytes,
                   file_name="pid_sim.csv", mime="text/csv")
```

**练习 4**：每次 widget 交互都会让整个脚本**从头重跑**，普通（全局）变量每次都被重新赋值，
无法记住上一次的状态；`st.session_state` 是 Streamlit 在重跑之间专门保留的存储区。

</details>

---

## 延伸阅读

- [Streamlit 官方文档（Get started）](https://docs.streamlit.io/get-started)
- [Streamlit API 参考：widgets](https://docs.streamlit.io/develop/api-reference/widgets)
- [st.session_state 详解](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state)
- [Streamlit 缓存（st.cache_data，进阶提速）](https://docs.streamlit.io/develop/concepts/architecture/caching)